# Differentiating through the solver

`pounce.jax.solve` wraps the solver in `jax.custom_vjp`. You can put it inside a JAX `loss(p)` and call `jax.grad(loss)(p)` to get derivatives **of the solution** $x^\star(p)$ with respect to problem parameters $p$.

Under the hood: the backward applies the **implicit function theorem** to the KKT system at $x^\star$. For

$$F(x^\star, p) = \nabla_x L(x^\star, \lambda^\star; p) = 0,$$

we have $\partial_p x^\star = -[\partial_x F]^{-1} \partial_p F$. We never differentiate through the iterates of the solver itself.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np

from pounce.jax import solve

## A trivial parametric QP — known closed form

Solve $\min_x \|x - p\|^2$. The optimum is $x^\star(p) = p$, so $\partial x^\star / \partial p = I$, and for any $L(p) = \sum_i x^\star_i(p)^2$ we expect $\nabla_p L = 2 p$.

Sanity-check the implicit-diff plumbing:

In [2]:
def f(x, p):
    d = x - p
    return jnp.dot(d, d)

def loss(p):
    x_star = solve(
        p, f=f, g=None, x0=jnp.zeros_like(p),
        n=p.size, m=0,
        options={"tol": 1e-10, "print_level": 0},
    )
    return jnp.sum(x_star ** 2)

p = jnp.array([1.0, -2.0, 3.0])
grad = jax.grad(loss)(p)
print(f"computed grad:    {np.asarray(grad)}")
print(f"closed-form 2p:   {np.asarray(2.0 * p)}")
print(f"max abs error:    {np.max(np.abs(grad - 2.0 * p)):.2e}")

ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable


computed grad:    [ 2. -4.  6.]
closed-form 2p:   [ 2. -4.  6.]
max abs error:    0.00e+00


## A practical use: hyperparameter learning

Suppose you've parametrized a regression: predict $y$ from features $\Phi$ by solving the ridge problem

$$w^\star(\alpha) = \arg\min_w \|y - \Phi w\|^2 + \alpha \|w\|^2.$$

Then evaluate validation MSE

$$L(\alpha) = \|y_{\text{val}} - \Phi_{\text{val}} w^\star(\alpha)\|^2$$

and tune $\alpha$ by gradient descent on $L$. The gradient $dL / d\alpha$ requires differentiating through the inner solve — which `pounce.jax.solve` makes a one-liner.

In [3]:
rng = np.random.default_rng(0)
n_features = 10
n_train, n_val = 60, 60
true_w = rng.standard_normal(n_features)
Phi_train = rng.standard_normal((n_train, n_features))
y_train = Phi_train @ true_w + 0.3 * rng.standard_normal(n_train)
Phi_val = rng.standard_normal((n_val, n_features))
y_val = Phi_val @ true_w + 0.3 * rng.standard_normal(n_val)

Phi_train_j = jnp.asarray(Phi_train)
y_train_j = jnp.asarray(y_train)
Phi_val_j = jnp.asarray(Phi_val)
y_val_j = jnp.asarray(y_val)

def inner_obj(w, alpha):
    r = y_train_j - Phi_train_j @ w
    return jnp.dot(r, r) + jnp.exp(alpha) * jnp.dot(w, w)

def val_loss(alpha):
    # alpha is a scalar (log-alpha to keep it >0 implicitly).
    p = jnp.atleast_1d(alpha)
    w_star = solve(
        p,
        f=lambda w, p: inner_obj(w, p[0]),
        g=None,
        x0=jnp.zeros(n_features),
        n=n_features, m=0,
        options={"tol": 1e-10, "print_level": 0},
    )
    r = y_val_j - Phi_val_j @ w_star
    return jnp.dot(r, r) / n_val

In [4]:
# Gradient descent on log-alpha.
alpha = jnp.array(0.0)
lr = 0.3
g = jax.grad(val_loss)

history = []
for step in range(30):
    L = float(val_loss(alpha))
    da = float(g(alpha))
    history.append((step, float(alpha), L, da))
    alpha = alpha - lr * da

for step, a, L, da in history[::3]:
    print(f"step {step:3d}: log-α = {a:+.3f}  L_val = {L:.4f}  dL/dα = {da:+.3e}")
print(f"\nfinal α = exp({float(alpha):.3f}) = {float(jnp.exp(alpha)):.3f}")

ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable


ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'N

ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'N

ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'N

ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'N

...

ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'N

ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'N

ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'N

step   0: log-α = +0.000  L_val = 0.1397  dL/dα = +3.302e-03
step   3: log-α = -0.003  L_val = 0.1397  dL/dα = +3.284e-03
step   6: log-α = -0.006  L_val = 0.1397  dL/dα = +3.266e-03
step   9: log-α = -0.009  L_val = 0.1396  dL/dα = +3.249e-03
step  12: log-α = -0.012  L_val = 0.1396  dL/dα = +3.231e-03
step  15: log-α = -0.015  L_val = 0.1396  dL/dα = +3.214e-03
step  18: log-α = -0.018  L_val = 0.1396  dL/dα = +3.197e-03
step  21: log-α = -0.020  L_val = 0.1396  dL/dα = +3.180e-03
step  24: log-α = -0.023  L_val = 0.1396  dL/dα = +3.163e-03
step  27: log-α = -0.026  L_val = 0.1396  dL/dα = +3.147e-03

final α = exp(-0.029) = 0.971


ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable
ERROR pounce::py: pounce-py: jacobian(): TypeError: 'NoneType' object is not callable


## Notes

* The solver is **single-threaded and stateful** (it owns a Rust `IpoptApplication` rebuilt per call). For batched parameter sweeps, `pounce.jax.vmap_solve` runs a sequential `jax.lax.map` rather than a true `vmap`.
* The forward call is wrapped in `jax.pure_callback`, so JAX sees the solver as a black-box. That's why you need to specify `n`, `m`, etc. up front — JAX needs the output shape ahead of time.
* Active-set handling in the backward. **Variable bounds**: any coordinate whose bound multiplier exceeds `_ACTIVE_TOL` is treated as pinned and contributes zero to $dx^\star/dp$. **General inequality rows** `cl[i] <= g_i(x) <= cu[i]`: a row is kept in the KKT block only if it is an equality (`cl[i] == cu[i]`) or its constraint multiplier `mult_g[i]` exceeds `_ACTIVE_TOL` (binding). Slack inequality rows are dropped — including them as equalities would silently return the wrong gradient (pounce#73).
* The Lagrangian Hessian used in the KKT solve comes from JAX (you don't need to supply one).